# Flight Delay Prediction — Phase 2 (Deep Learning)

## 1. Setup

In [1]:
import os
# PyTorch and XGBoost each ship their own OpenMP library; on macOS they crash if both start
# multi-threaded OpenMP. One OpenMP thread avoids this (PyTorch still uses 4 threads below).
os.environ["OMP_NUM_THREADS"] = "1"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from nycflights13 import flights
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_recall_curve, roc_curve)

sns.set_style("whitegrid")
os.makedirs("figures", exist_ok=True)

RANDOM_STATE = 42
torch.set_num_threads(4)   # CPU training is fast enough for this data and gives repeatable results
print("PyTorch", torch.__version__)

PyTorch 2.15.0


## 2. Data and features (same pipeline as Phase 1)

In [2]:
df = flights.copy()

# Drop cancelled / diverted flights (no arrival delay), build the target and a date column
df = df.dropna(subset=["arr_delay"]).copy()
df["delayed"] = (df["arr_delay"] >= 15).astype(int)
df["date"] = pd.to_datetime(df[["year", "month", "day"]])

# Remove everything only known after departure (data leakage) and unused ID / duplicate columns
leaky_columns = ["dep_time", "dep_delay", "arr_time", "arr_delay", "air_time"]
df = df.drop(columns=leaky_columns + ["tailnum", "flight", "time_hour", "year", "day", "hour", "minute"])

print("Flights:", len(df), " delayed share:", round(df["delayed"].mean(), 3))

Flights: 327346  delayed share: 0.245


In [3]:
# Calendar and schedule features with cyclic (sin/cos) encoding
df["day_of_week"] = df["date"].dt.dayofweek
df["dep_hour"] = df["sched_dep_time"] // 100
df["arr_hour"] = df["sched_arr_time"] // 100
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

def add_cyclic(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

add_cyclic(df, "dep_hour", 24)
add_cyclic(df, "arr_hour", 24)
add_cyclic(df, "day_of_week", 7)

# Airport congestion, counted on the FULL schedule (including flights that were later cancelled)
schedule = flights.copy()
schedule["date"] = pd.to_datetime(schedule[["year", "month", "day"]])
schedule["dep_hour"] = schedule["sched_dep_time"] // 100
hour_counts = schedule.groupby(["date", "origin", "dep_hour"]).size().rename("origin_hour_flights")
day_counts = schedule.groupby(["date", "origin"]).size().rename("origin_day_flights")
df = df.join(hour_counts, on=["date", "origin", "dep_hour"]).join(day_counts, on=["date", "origin"])

In [4]:
def add_past_delay_rate(df, col, window=7):
    """Share of delayed flights for each value of `col` over the previous `window` days (today excluded)."""
    delayed_per_day = df.pivot_table(index="date", columns=col, values="delayed", aggfunc="sum", fill_value=0)
    flights_per_day = df.pivot_table(index="date", columns=col, values="delayed", aggfunc="count", fill_value=0)
    past_delayed = delayed_per_day.rolling(window, min_periods=1).sum().shift(1)
    past_flights = flights_per_day.rolling(window, min_periods=1).sum().shift(1)
    rate = past_delayed / past_flights.replace(0, np.nan)
    rate = rate.stack().rename(f"{col}_delay_rate_7d").reset_index()
    return df.merge(rate, on=["date", col], how="left")

df["route"] = df["origin"] + "-" + df["dest"]
for col in ["carrier", "origin", "dest", "route"]:
    df = add_past_delay_rate(df, col)

numeric_features = ["is_weekend", "day_of_week_sin", "day_of_week_cos",
                    "dep_hour_sin", "dep_hour_cos", "arr_hour_sin", "arr_hour_cos", "distance",
                    "origin_hour_flights", "origin_day_flights",
                    "carrier_delay_rate_7d", "origin_delay_rate_7d", "dest_delay_rate_7d", "route_delay_rate_7d"]
history_features = [c for c in numeric_features if "delay_rate" in c]
categorical_features = ["carrier", "origin", "dest"]

assert not set(leaky_columns) & set(numeric_features + categorical_features)

## 3. Train / validation / test split by date

In [5]:
train = df[df["month"] <= 8].reset_index(drop=True)          # Jan-Aug
val = df[df["month"].isin([9, 10])].reset_index(drop=True)    # Sep-Oct
test = df[df["month"] >= 11].reset_index(drop=True)           # Nov-Dec
assert train["date"].max() < val["date"].min() and val["date"].max() < test["date"].min()

y_train, y_val, y_test = train["delayed"].values, val["delayed"].values, test["delayed"].values

pd.DataFrame({"flights": [len(train), len(val), len(test)],
              "delayed_share": [y_train.mean(), y_val.mean(), y_test.mean()]},
             index=["train", "validation", "test"]).round(3)

,flights,delayed_share
train,217727,0.264
validation,55628,0.157
test,53991,0.258


## 4. Phase 1 baselines (Logistic Regression and XGBoost)

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier

def one_hot_features(data):
    return pd.get_dummies(data[numeric_features + categorical_features], columns=categorical_features, dtype=int)

X_train = one_hot_features(train)
X_val = one_hot_features(val).reindex(columns=X_train.columns, fill_value=0)
X_test = one_hot_features(test).reindex(columns=X_train.columns, fill_value=0)
train_means = X_train[history_features].mean()
X_train, X_val, X_test = [X.fillna(train_means) for X in (X_train, X_val, X_test)]

# Same settings as Phase 1 (XGBoost depth 3 and 100 trees were chosen there by time-series validation)
log_reg = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(X_train, y_train)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                    min_child_weight=50, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
                    random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

test_prob = {"Logistic Regression": log_reg.predict_proba(X_test)[:, 1], "XGBoost": xgb.predict_proba(X_test)[:, 1]}
val_prob = {"Logistic Regression": log_reg.predict_proba(X_val)[:, 1], "XGBoost": xgb.predict_proba(X_val)[:, 1]}

# Phase 1 calibration (isotonic regression fitted on validation) for the XGBoost baseline
iso_xgb = IsotonicRegression(out_of_bounds="clip").fit(val_prob["XGBoost"], y_val)
xgb_test_cal = iso_xgb.predict(test_prob["XGBoost"])

for name, p in test_prob.items():
    print(f"{name:20s} test PR AUC {average_precision_score(y_test, p):.4f}  ROC AUC {roc_auc_score(y_test, p):.4f}")

Logistic Regression  test PR AUC 0.3761  ROC AUC 0.6451
XGBoost              test PR AUC 0.3908  ROC AUC 0.6560


## 5. Preparing the data for the neural network

In [7]:
# Categories -> integer ids for the embedding layers. Id 0 is reserved for values never seen in training.
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(train[c].unique()))} for c in categorical_features}
vocab_sizes = {c: len(vocab[c]) + 1 for c in categorical_features}

# Numeric features: fill missing history with the training mean, then standardise with training statistics
num_means = train[numeric_features].mean()
num_stds = train[numeric_features].std()

def make_tensors(data, num_cols=numeric_features, cat_cols=categorical_features):
    x_num = ((data[num_cols].fillna(num_means[num_cols]) - num_means[num_cols]) / num_stds[num_cols]).values
    if cat_cols:
        x_cat = np.stack([data[c].map(vocab[c]).fillna(0).astype(int).values for c in cat_cols], axis=1)
    else:
        x_cat = np.zeros((len(data), 0), dtype=int)
    return (torch.tensor(x_num, dtype=torch.float32), torch.tensor(x_cat, dtype=torch.long),
            torch.tensor(data["delayed"].values, dtype=torch.float32))

train_t, val_t, test_t = make_tensors(train), make_tensors(val), make_tensors(test)

print("Numeric inputs:", train_t[0].shape[1], " categorical inputs:", train_t[1].shape[1])
print("Vocabulary sizes (incl. unseen id 0):", vocab_sizes)
print("Unseen destinations in test:", int((test_t[1][:, 2] == 0).sum()), "flights")

Numeric inputs: 14  categorical inputs: 3
Vocabulary sizes (incl. unseen id 0): {'carrier': 17, 'origin': 4, 'dest': 103}
Unseen destinations in test: 57 flights


## 6. Model: MLP with learned embeddings

In [8]:
EMBED_DIMS = {"carrier": 8, "origin": 2, "dest": 16}   # roughly the 4th root of the number of categories, x 4

class DelayNet(nn.Module):
    """Embeddings for carrier / origin / destination + numeric features -> 2 hidden layers -> delay logit.

    mode = "embedding": learned dense vectors (the Phase 2 model)
    mode = "onehot":    one-hot vectors instead of embeddings (ablation)
    mode = "none":      no categorical inputs at all (ablation)
    """
    def __init__(self, n_numeric, cat_cols=categorical_features, mode="embedding", hidden=(128, 64), dropout=0.3):
        super().__init__()
        self.mode, self.cat_cols = mode, cat_cols
        if mode == "embedding":
            self.embeddings = nn.ModuleList([nn.Embedding(vocab_sizes[c], EMBED_DIMS[c]) for c in cat_cols])
            n_in = n_numeric + sum(EMBED_DIMS[c] for c in cat_cols)
        elif mode == "onehot":
            n_in = n_numeric + sum(vocab_sizes[c] for c in cat_cols)
        else:
            n_in = n_numeric
        layers = []
        for h in hidden:
            layers += [nn.Linear(n_in, h), nn.ReLU(), nn.Dropout(dropout)]
            n_in = h
        layers.append(nn.Linear(n_in, 1))      # one output: the logit of P(delayed)
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        parts = [x_num]
        if self.mode == "embedding":
            parts += [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        elif self.mode == "onehot":
            parts += [F.one_hot(x_cat[:, i], vocab_sizes[c]).float() for i, c in enumerate(self.cat_cols)]
        return self.mlp(torch.cat(parts, dim=1)).squeeze(1)

model = DelayNet(n_numeric=len(numeric_features))
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

DelayNet(
  (embeddings): ModuleList(
    (0): Embedding(17, 8)
    (1): Embedding(4, 2)
    (2): Embedding(103, 16)
  )
  (mlp): Sequential(
    (0): Linear(in_features=40, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)
Trainable parameters: 15361


In [9]:
def predict_logits(model, data):
    model.eval()
    with torch.no_grad():
        return model(data[0], data[1]).numpy()

def predict_proba(model, data):
    return 1 / (1 + np.exp(-predict_logits(model, data)))

def train_model(train_data, val_data, seed=0, mode="embedding", dropout=0.3, weight_decay=1e-4,
                cat_cols=categorical_features, lr=1e-3, batch_size=1024, max_epochs=30, patience=5):
    """Mini-batch training with AdamW and early stopping on validation PR AUC."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = DelayNet(train_data[0].shape[1], cat_cols=cat_cols, mode=mode, dropout=dropout)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()           # binary cross-entropy on the logit
    x_num, x_cat, y = train_data
    history, best_score, best_state, epochs_without_gain = [], -1, None, 0

    for epoch in range(max_epochs):
        model.train()
        order = torch.randperm(len(y))
        total_loss = 0.0
        for start in range(0, len(y), batch_size):
            idx = order[start:start + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x_num[idx], x_cat[idx]), y[idx])
            loss.backward()                    # backpropagation
            optimizer.step()
            total_loss += loss.item() * len(idx)
        val_pr = average_precision_score(val_data[2].numpy(), predict_proba(model, val_data))
        history.append({"epoch": epoch + 1, "train_loss": total_loss / len(y), "val_PR_AUC": val_pr})

        if val_pr > best_score:                # early stopping: keep the best epoch, stop after `patience` bad ones
            best_score, epochs_without_gain = val_pr, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_gain += 1
            if epochs_without_gain >= patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)